In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.preprocessing import normalize, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from __future__ import print_function
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import mean_squared_error
import warnings
import gc, sys
warnings.filterwarnings('ignore')

In [ ]:
url = "C:/Users/LAPTOP/Downloads/yellow_tripdata_2019-06.csv"
taxi_data = pd.read_csv(url)
taxi_data.head(1000)


In [ ]:
taxi_data = taxi_data[taxi_data['tip_amount'] > 0]
taxi_data = taxi_data[(taxi_data['tip_amount'] <= taxi_data['fare_amount'])]
taxi_data = taxi_data[((taxi_data['fare_amount'] >= 2) & (taxi_data['fare_amount'] < 200))]
cleaned_data = taxi_data.drop(['total_amount'], axis = 1)
del taxi_data
gc.collect()
print("There are " + str(len(cleaned_data)) + " observations in the dataset.")
print("There are " + str(len(cleaned_data.columns)) + " variables in the dataset.")
plt.hist(cleaned_data.tip_amount.values, bins = 100, histtype ='bar', color ='r')
plt.show()
print("Minimum tip amount  is ", np.min(cleaned_data.tip_amount.values))
print("Maximum tip amount is ", np.max(cleaned_data.tip_amount.values))
print("90% of the trips have a tip amount less or equal than ", np.percentile(cleaned_data.tip_amount.values, 90))

In [ ]:
cleaned_data.head()

In [ ]:
cleaned_data['tpep_dropoff_datetime'] = pd.to_datetime(cleaned_data['tpep_dropoff_datetime'])
cleaned_data['tpep_pickup_datetime'] = pd.to_datetime(cleaned_data['tpep_pickup_datetime'])
cleaned_data['pickup_hour'] = cleaned_data['tpep_pickup_datetime'].dt.hour
cleaned_data['dropoff_hour'] = cleaned_data['tpep_dropoff_datetime'].dt.hour
cleaned_data['pickup_day'] = cleaned_data['tpep_pickup_datetime'].dt.weekday
cleaned_data['dropoff_day'] = cleaned_data['tpep_dropoff_datetime'].dt.weekday
cleaned_data['trip_time'] = (cleaned_data['tpep_dropoff_datetime'] - cleaned_data['tpep_pickup_datetime']).dt.total_seconds()
n_num_rows = 200000
cleaned_data = cleaned_data.head(n_num_rows)

In [ ]:
cleaned_data = cleaned_data.drop(columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime'], axis = 1)

In [ ]:
get_dummy_col = ["VendorID", "RatecodeID", "store_and_fwd_flag", "PULocationID", "DOLocationID", "payment_type", "pickup_hour", "dropoff_hour", "pickup_day", "dropoff_day"]
proc_data = pd.get_dummies(cleaned_data, columns=get_dummy_col)
del cleaned_data
gc.collect()

In [ ]:
y_variable = proc_data[['tip_amount']].values.astype('float32')
proc_data = proc_data.drop(['tip_amount'], axis=1)
x_variable = proc_data.values
x_variable = normalize(x_variable, axis=1, norm='l1', copy=False)

In [ ]:
print(x_variable.shape, y_variable.shape)


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_variable, y_variable, test_size = 0.3, random_state = 40)
reg_tree = DecisionTreeRegressor(max_depth = 6, random_state = 32)
now = time.time()
reg_tree.fit(x_train, y_train)
later = time.time()-now
print(later)
y_pred = reg_tree.predict(x_test)
print(y_pred) 

In [ ]:
from snapml import DecisionTreeRegressor
reg_snap = DecisionTreeRegressor(max_depth = 6, n_jobs = 4, random_state = 35)
initial = time.time()
reg_snap.fit(x_train, y_train)
final = time.time() - initial
print(final)